# D38 — FULL 90 câu bằng **ReAct + Calculator tool** (Qwen3-4B)

Dạng D38: *phương trình trộn $z$ với $|z|$ (ví dụ $|z|(z-a)+bi=cz$), đếm số
số phức z thỏa mãn*. Bản chạy đầy đủ của phương pháp đã kiểm chứng tay
nhiều câu trong `KLTN_D38_ReAct_Calculator_1cau.ipynb`, bao gồm 10 câu khó
nhất (chọn theo tiêu chí cả Gemini lẫn Qwen3-4B zero-shot đều thất bại,
cộng thêm các câu hiếm có đáp số khác 1).

Model **không tự tính tay bất kỳ phép nào** — mọi phép tính đều gọi tool
`Calculator` (sympy, chính xác tuyệt đối, **có bộ nhớ biến**, hỗ trợ
**ẩn số tự do** `{'t'}` để t=|z| có thể tồn tại như ẩn chưa biết
xuyên suốt phần lớn quá trình, và hỗ trợ `nroots()` cho đa thức bậc cao
không có nghiệm đóng đơn giản, giống chế độ giải phương trình số của máy
tính Casio thật) theo vòng lặp ReAct thật:
`Thought → Action → Action Input → Observation → …`

Prompt được **ép mở đầu bằng `<think>\nThought:`** (forced prefix) —
model không còn quyền tự chọn viết văn xuôi mở đầu trước khi vào định dạng
ReAct, đúng bản sửa lỗi cuối cùng đã kiểm chứng ổn định trên nhiều câu liên
tiếp trong bản 1 câu.

## Điểm khác bản 1 câu: vòng lặp ReAct chạy THEO LÔ

Chạy tuần tự 90 câu sẽ mất hàng giờ. Ở đây mỗi **vòng** gọi vLLM **một lần**
cho tất cả các câu đang hoạt động (vLLM tự batching), rồi chạy Calculator
riêng cho từng câu, rồi generate tiếp.

Mỗi câu có **bộ nhớ biến riêng** (`MayTinh()` riêng), **ngân sách token
riêng**, và tự thoát khỏi lô khi viết xong `Final Answer`.

Hạn mức tool gọi/câu được nâng lên so với các dạng khác (`35          # van an toan chong loop vo han (D38 can nhieu buoc hon: toi da 4 nghiem, moi nghiem toi da 4 lan goi tool)`)
vì D38 có thể cần lọc tới 4 nghiệm, mỗi nghiệm tới 4 lượt gọi tool (kiểm
tra là số thực, kiểm tra ngưỡng $t\ge0$, khôi phục $z$, xác minh tự hợp).

## Prompt & backend giống hệt bản 1 câu

Cell 4 (prompt) và phần backend `MayTinh` ở Cell 5 được **trích nguyên văn**
từ notebook 1 câu bằng script `scratch/build_react_full_D38.py` — không gõ
lại, nên không có nguy cơ lệch giữa 2 bản.

## Trước khi chạy

Upload `plan_solve_prompts_D38.json` (90 câu, đã sinh sẵn từ
`Sinh_them_cau_hoi/So_phuc_day_du.csv`, đã loại câu gốc STT 3302 nằm trong
few-shot) thành Kaggle Dataset (slug gợi ý `d38-full90`), gắn vào notebook,
bật GPU.

Kết quả: `/kaggle/working/d38_react_calculator_full90.csv`

In [ ]:
!pip install -q -U vllm
!pip uninstall -y -q torchcodec
import vllm; print('vLLM:', vllm.__version__)

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

DATA_PATH = '/kaggle/input/d38-full90/plan_solve_prompts_D38.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d38_react_calculator_full90.csv'

MODEL = 'Qwen/Qwen3-4B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.3, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

AN_SO_TU_DO = {'t'}

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(DATA_PATH, encoding='utf-8') as f:
    records = json.load(f)
print('So cau:', len(records))

tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype='float16',
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90,
          trust_remote_code=True, enforce_eager=True, seed=SEED)
print('San sang.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D38_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai [C] PART_HUONGGIAI va
# [D] PART_FEWSHOT, giu nguyen [A] PART_TOOL va [B] PART_KIENTHUC.

# [A] PART_TOOL - HUONG DAN DUNG TOOL (DUNG CHUNG CHO MOI DANG TOAN)
# =====================================================================
PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- To test whether two expressions are **exactly** equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`, decided exactly; never approximately.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Some equations genuinely have no simple closed form, such as a cubic, quartic, or higher-degree polynomial that does not factor into nice roots: for those, `nroots(<polynomial>)` gives the Calculator's own numeric roots, exactly like a real handheld calculator's equation-solve mode; this is still the Calculator computing, not you approximating. `nroots(...)` returns EVERY root of the polynomial, including non-real ones when the polynomial's real roots don't account for its full degree; before doing anything else with a root, check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`; `Im` is not recognized and silently fails to evaluate) and discard it immediately if not. For a root confirmed real, use `re(root)`; not the raw value; in every later threshold or substitution, since even a numerically-real root can carry a residual non-zero imaginary part too small to matter but large enough to break a direct comparison. Any comparison built on such numeric roots (a threshold like `t > 0`, or a self-consistency check like `Abs(a - b) < 1e-6`) should use a small tolerance instead of exact `Eq()`; everything that does not depend on a numeric root; in particular the final count of solutions and matching it against the answer options; stays exact as usual.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


# =====================================================================
# [B] PART_KIENTHUC - KIEN THUC NEN SO PHUC (DUNG CHUNG MOI DANG SO PHUC)
# =====================================================================
PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
'''


# =====================================================================
# [C] PART_HUONGGIAI - HUONG GIAI RIENG CUA DANG D38 (THAY KHI DOI DANG)
# =====================================================================
PART_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE**

Problem shape: an equation mixes the unknown $z$ itself with its modulus $|z|$ in several places (e.g. $|z|(z-a)+bi=cz$). Count how many complex numbers $z$ satisfy it.

Follow this method exactly; do not invent a shorter route.

1. **Isolate $z$.** Expand and move every term containing $z$ to one side, everything else to the other side. Factor out $z$: the equation becomes $z\cdot(\text{Expr}_1) = \text{Expr}_2$, where both $\text{Expr}_1$ and $\text{Expr}_2$ may still contain $|z|$.
2. **Substitute $t=|z|$**, a real unknown with $t\ge0$ (do not assign it a value yet; it stays symbolic, like $x,y$ did in other problem types, until solved for near the end). $\text{Expr}_1$ and $\text{Expr}_2$ are now complex expressions in $t$; call them $\text{bt}_1(t)$ and $\text{bt}_2(t)$.
3. **Take the modulus of both sides.** Since $|z\cdot w|=|z|\cdot|w|$ and $|z|=t$: $t\cdot|\text{bt}_1(t)| = |\text{bt}_2(t)|$. Square both sides (both sides are already non-negative here, so squaring introduces no extraneous roots) to remove the square roots inside each modulus: $t^2\cdot|\text{bt}_1(t)|^2 = |\text{bt}_2(t)|^2$. Compute $|\text{bt}_1(t)|^2$ and $|\text{bt}_2(t)|^2$ with the Calculator (`Abs(...)**2`), then expand the full equation into a single polynomial in $t$ set to $0$.
4. **Solve the polynomial for $t$.** This is usually degree 4 (sometimes higher if the given constants are irrational). Use `nroots(...)` to get all numeric roots; this returns every root of the polynomial, real or not. Then initialize `count = 0` with the Calculator; this running tally is the only place `count` ever gets set; every later value it takes comes from the Calculator adding to it, never from a number you recall.
5. **Filter.** For each root in turn, first check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`) and discard it immediately if not; do not test it against $t\ge0$ or go any further with it. For a root confirmed real, use `re(root)`; not the raw value; for everything from here on, and check it is $\ge0$ with a numeric `> 0` comparison (not exact `Eq()`, since these are decimal values); discard it if that fails.
6. **Recover $z$ and verify self-consistency.** For each root that survives step 5, compute $z = \text{bt}_2(t)/\text{bt}_1(t)$ by substituting the numeric $t$, then verify $|z|$ actually equals that same $t$ using a small-tolerance check (`Abs(Abs(z) - t) < 1e-6`); this is a free safety net against any arithmetic slip made while building the polynomial in step 3, costing nothing since the Calculator is doing the check, not you. The moment a root passes this check, increment the tally with the Calculator: `count = count + 1`. If it fails, discard the root and leave `count` unchanged.
7. **Match.** Once every root from `nroots(...)` has been processed this way, `count` already holds the exact number of complex numbers $z$ satisfying the problem. Test the four options in order with `Eq(count, <option>)` and stop at the first `True`; never pick the letter by eye, and never re-declare `count` to a number you recall instead of the running tally above.
'''


# =====================================================================
# [D] PART_FEWSHOT - VI DU MAU RIENG CUA DANG D38 (THAY KHI DOI DANG)
# Cac dong "Observation:" duoi day la KET QUA THAT (chay bang chinh class
# MayTinh o backend, khong phai sympy roi rac va khong phai so bia).
# =====================================================================
PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Có bao nhiêu số phức $z$ thỏa mãn $|z|(z+6+i)-5i=(5+5i)z$?
A. $0$ | B. $1$ | C. $2$ | D. $3$

<think>
Thought: Isolate z one term at a time; this is the step most likely to go wrong, so slow down and do not skip listing anything. Original: |z|(z+6+i)-5i=(5+5i)z, with t=|z|. Expand every product first: t*z + 6t + t*i - 5i = 5z + 5*i*z. Now list every single term and whether it contains z: left side has t*z (has z), +6t (no z), +t*i (no z), -5i (no z); right side has 5z (has z), +5*i*z (has z). Move the two z-terms from the right to the left, flipping each one's sign as it crosses: t*z - 5z - 5*i*z. Move the three non-z terms from the left to the right, flipping each one's sign as it crosses: -6t - t*i + 5i. So the equation is now t*z - 5z - 5*i*z = -6t - t*i + 5i. Factor z out of the left side: z*(t-5-5i) = -6t - t*i + 5i.
Thought: OK, the coefficient of z is t-5-5i. I will now type exactly: expr1 = t - 5 - 5*I
Action: Calculator
Action Input: expr1 = t - 5 - 5*I
Observation: expr1 = t - 5 - 5*I

Thought: OK, grouping the right side into real and imaginary parts gives -6t + (5-t)i. I will now type exactly: expr2 = -6*t + (5-t)*I
Action: Calculator
Action Input: expr2 = -6*t + (5-t)*I
Observation: expr2 = -6*t - I*t + 5*I

Thought: Compute |expr1|^2.
Action: Calculator
Action Input: mod_expr1_sq = Abs(expr1)**2
Observation: mod_expr1_sq = t**2 - 10*t + 50

Thought: Compute |expr2|^2.
Action: Calculator
Action Input: mod_expr2_sq = Abs(expr2)**2
Observation: mod_expr2_sq = 37*t**2 - 10*t + 25

Thought: Build the polynomial t^2*|expr1|^2 - |expr2|^2 = 0.
Action: Calculator
Action Input: poly = expand(t**2*mod_expr1_sq - mod_expr2_sq)
Observation: poly = t**4 - 10*t**3 + 13*t**2 + 10*t - 25

Thought: This is degree 4 and does not factor into simple exact roots, so solve it numerically.
Action: Calculator
Action Input: roots_t = nroots(poly)
Observation: roots_t = [-1.18475798583764, 8.34072230366137, 1.42201784108813 - 0.71259017413456*I, 1.42201784108813 + 0.71259017413456*I]

Thought: Start the running tally at 0 before checking any root.
Action: Calculator
Action Input: count = 0
Observation: count = 0

Thought: Check the first root: is it real before doing anything else with it.
Action: Calculator
Action Input: t1 = roots_t[0]
Observation: t1 = -1.18475798583764

Action: Calculator
Action Input: is_real1 = Abs(im(t1)) < 1e-9
Observation: is_real1 = True

Thought: t1 is real, so now test it against t>=0.
Action: Calculator
Action Input: t1 > 0
Observation: False

Thought: t1 is negative, discarded. Check the second root.
Action: Calculator
Action Input: t2 = roots_t[1]
Observation: t2 = 8.34072230366137

Action: Calculator
Action Input: is_real2 = Abs(im(t2)) < 1e-9
Observation: is_real2 = True

Thought: t2 is real. Test t2 > 0.
Action: Calculator
Action Input: t2 > 0
Observation: True

Thought: t2 passes the threshold. Recover z2 = expr2(t2)/expr1(t2).
Action: Calculator
Action Input: z2 = expr2.subs(t, t2) / expr1.subs(t, t2)
Observation: z2 = -4.16147234247151 - 7.22840207028074*I

Thought: Verify self-consistency: |z2| should equal t2.
Action: Calculator
Action Input: check2 = Abs(Abs(z2) - t2) < 1e-6
Observation: check2 = True

Thought: t2 passed both checks; increment the tally.
Action: Calculator
Action Input: count = count + 1
Observation: count = 1

Thought: t2 is valid. Check the third root; test its reality first, before any threshold or substitution.
Action: Calculator
Action Input: t3 = roots_t[2]
Observation: t3 = 1.42201784108813 - 0.71259017413456*I

Action: Calculator
Action Input: is_real3 = Abs(im(t3)) < 1e-9
Observation: is_real3 = False

Thought: t3 is not real; discard it immediately, no need to test t>0 or recover z from it. Check the fourth root the same way.
Action: Calculator
Action Input: t4 = roots_t[3]
Observation: t4 = 1.42201784108813 + 0.71259017413456*I

Action: Calculator
Action Input: is_real4 = Abs(im(t4)) < 1e-9
Observation: is_real4 = False

Thought: t4 is not real either; discard it too. Every root has now been processed; the running tally already holds the final count. Read option A verbatim from the problem, one option at a time: "A. $0$"; A is 0. Test it.
Action: Calculator
Action Input: Eq(count, 0)
Observation: False

Thought: Not A. Read option B verbatim: "B. $1$"; B is 1. Test it.
Action: Calculator
Action Input: Eq(count, 1)
Observation: True

Option B matches; stop here.
</think>
Final Answer: \boxed{B}
'''


# =====================================================================
# [E] PART_NHIEMVU - CHOT NHIEM VU (sua danh sach buoc khi doi dang)
# =====================================================================
PART_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] Rearrange the given equation to isolate $z$; this is the step most likely to go wrong, so do not skip any part of it in your Thought. Expand every product first. Then list every single term on both sides and mark whether each one contains $z$. Move the $z$-terms to one side one at a time, flipping each one's sign as it crosses the equals sign; then move the non-$z$-terms to the other side the same way, one term at a time. Once every term has been accounted for, factor $z$ out and group the other side into real and imaginary parts. Then, right before each of the two Calculator calls, write a Thought that spells out the EXACT line you are about to type in Calculator syntax (e.g. "I will now type exactly: expr1 = ..."); then your `Action Input` must copy that line character-for-character, not retype it from the earlier prose. This turns the risky step into a copy instead of a fresh translation, which is exactly what protects against mixing up two similar-looking coefficients (e.g. two different radicals) in the same equation.
[Step 2] Compute `mod_expr1_sq = Abs(expr1)**2` and `mod_expr2_sq = Abs(expr2)**2`.
[Step 3] Compute `poly = expand(t**2*mod_expr1_sq - mod_expr2_sq)`.
[Step 4] Compute `roots_t = nroots(poly)`, then set `count = 0`; this is the only place `count` is ever set directly; from here it only changes via `count = count + 1`.
[Step 5] For each root: extract it, check `Abs(im(root)) < 1e-9` and discard immediately if not real; otherwise take `re(root)` and test it `> 0` (discard if not).
[Step 6] For each root that survives step 5, compute the matching $z$ via substitution into `expr2/expr1` and verify self-consistency with `Abs(Abs(z) - t_i) < 1e-6`; if it passes, increment with `count = count + 1`, if it fails, discard the root and leave `count` unchanged.
[Step 7] Once every root has been processed, go through the options ONE AT A TIME in order. For each one, quote its exact text from the problem verbatim in your Thought before assigning it a number (e.g. read "A. $0$" and write A is 0); do not read ahead to other options or summarize several of them together, since that is what causes letters and values to get mixed up. Then test that one value with `Eq(count, <that value>)`, and stop at the first `True`; never re-declare `count` to a number you recall.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)


# BACKEND: TOOL "Calculator" - may tinh sympy CO BO NHO BIEN

In [ ]:
# BACKEND: TOOL "Calculator" - may tinh sympy CO BO NHO BIEN
# =====================================================================
# Tong quat cho MOI dang toan (khong chi D53): nhan 1 bieu thuc sympy bat
# ky, tra ve gia tri chinh xac tuyet doi. Ho tro:
#   - gan bien:  "ten = bieu_thuc"  -> luu vao bo nho, dung lai o luot sau
#     (xoa han lo hoi model chep tay lai so dai - nguon loi lon nhat)
#   - so khop :  "Eq(a, b)"         -> True/False chinh xac, khong xap xi
#   - moi phep cong tru nhan chia phan so, can thuc, so phuc, mo dun, lien hop
import sympy as sp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong (khong
        bao loi) -> ket qua se vo nghia ma model khong he hay biet. Chan lai
        moi ten KHONG nam trong bo nho VA khong nam trong danh sach an so
        tu do duoc khai bao truoc."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        if not ten_thieu:
            return None
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            parsed = sp.sympify(s, locals=self.ns)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau = sp.simplify(parsed.lhs - parsed.rhs) == 0
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri = sp.expand(sp.simplify(parsed))
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'


# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 35          # van an toan chong loop vo han (D38 can nhieu buoc hon: toi da 4 nghiem, moi nghiem toi da 4 lan goi tool)
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=AN_SO_TU_DO)   # bo nho + an so RIENG cho tung cau
        # EP SAN token dau tien (giong ban 1 cau): model bi buoc noi tiep tu
        # "Thought:" ngay sau <think>, khong con quyen tu chon viet van xuoi
        # mo dau (hanh vi mac dinh de lech khoi dinh dang ReAct).
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        if o.stop_reason != STOP_STR:
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        s.n_calls += 1
        n_goi_vong_nay += 1
        khop = None
        for mm in ACTION_INPUT_RE.finditer(o.text):
            khop = mm
        bieu_thuc = khop.group(1).strip() if khop else ''

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
print()
print('Da luu:', OUT_PATH)